In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ompardh/sentiment140/training.1600000.processed.noemoticon.csv


In [11]:
!pip install faiss-cpu
!pip install sentence-transformers
!pip install scikit-learn
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
import numpy as np
import re
import logging

DATA_PATH = "/kaggle/input/datasets/ompardh/sentiment140/training.1600000.processed.noemoticon.csv"
POLICY_KEYWORD = "bank"
MAX_SAMPLES = 10000

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

logger.info("Loading dataset...")

df_full = pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    header=None
)

df_full.columns = ["sentiment", "id", "date", "flag", "user", "text"]

logger.info(f"Total dataset size: {len(df_full)}")

df_filtered = df_full[
    df_full["text"].str.contains(POLICY_KEYWORD, case=False, na=False)
]

if len(df_filtered) == 0:
    raise ValueError("Keyword not found in dataset.")

df = df_filtered.head(MAX_SAMPLES).copy()

logger.info(f"Working dataset size: {len(df)}")


INFO:__main__:Loading dataset...
INFO:__main__:Total dataset size: 1600000
INFO:__main__:Working dataset size: 2317


In [7]:
def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.lower().strip()

df["clean_text"] = df["text"].apply(clean_text)

df = df[df["clean_text"].str.strip().astype(bool)]


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Map original labels
df["sentiment"] = df["sentiment"].map({0: "negative", 4: "positive"})

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X = vectorizer.fit_transform(df["clean_text"])
y = df["sentiment"]

sentiment_model = LogisticRegression(max_iter=1000)
sentiment_model.fit(X, y)


def sentiment_summary():
    probs = sentiment_model.predict_proba(X)
    max_probs = np.max(probs, axis=1)
    raw_preds = sentiment_model.predict(X)

    final_preds = []

    for i, prob in enumerate(max_probs):
        if prob < 0.60:   # 👈 THRESHOLD FOR NEUTRAL
            final_preds.append("neutral")
        else:
            final_preds.append(raw_preds[i])

    final_preds = np.array(final_preds)

    pos_ratio = np.mean(final_preds == "positive")
    neg_ratio = np.mean(final_preds == "negative")
    neutral_ratio = np.mean(final_preds == "neutral")

    confidence = np.mean(max_probs)

    return {
        "positive_ratio": pos_ratio,
        "negative_ratio": neg_ratio,
        "neutral_ratio": neutral_ratio,
        "confidence": confidence
    }


In [9]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(X)

def get_topics(model, feature_names, n_top_words=8):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_features = [
            feature_names[i]
            for i in topic.argsort()[:-n_top_words - 1:-1]
        ]
        topics.append(top_features)
    return topics

feature_names = vectorizer.get_feature_names_out()
topics = get_topics(lda, feature_names)

def topic_exploration():
    topic_examples = []
    topic_distribution = lda.transform(X)
    dominant_topics = topic_distribution.argmax(axis=1)
    
    for i in range(3):
        example_indices = np.where(dominant_topics == i)[0][:2]
        examples = df.iloc[example_indices]["text"].tolist()
        topic_examples.append({
            "topic_id": i,
            "keywords": topics[i],
            "examples": examples
        })
    
    return topic_examples


In [16]:
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(df["clean_text"].tolist())

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))


INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/README.md "HTTP/1.1 200 OK"


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/73 [00:00<?, ?it/s]

In [17]:
def retrieve(query, k=5):
    query_vector = embed_model.encode([query])
    distances, indices = index.search(np.array(query_vector), k)
    
    results = df.iloc[indices[0]][["text"]].copy()
    results["distance"] = distances[0]
    
    return results


In [18]:
class PolicyAgent:
    
    def route(self, query):
        query_lower = query.lower()
        
        if "sentiment" in query_lower:
            return sentiment_summary()
        
        elif "topic" in query_lower:
            return topic_exploration()
        
        else:
            return retrieve(query)

agent = PolicyAgent()


In [19]:
def generate_executive_report():
    sentiment_data = sentiment_summary()
    topic_data = topic_exploration()
    
    report = f"""
    Executive Policy Intelligence Report
    -------------------------------------
    Policy Keyword: {POLICY_KEYWORD}

    Sentiment Distribution:
    Positive: {sentiment_data['positive_ratio']:.2%}
    Negative: {sentiment_data['negative_ratio']:.2%}

    Key Topics Identified:
    """

    for topic in topic_data:
        report += f"\nTopic {topic['topic_id']}: {', '.join(topic['keywords'])}"
    
    report += f"""

    Risk Insight:
    Elevated negative sentiment around {POLICY_KEYWORD} may signal
    financial anxiety or trust concerns.

    Confidence Score: {sentiment_data['confidence']:.2f}
    """

    return report

print(generate_executive_report())



    Executive Policy Intelligence Report
    -------------------------------------
    Policy Keyword: bank

    Sentiment Distribution:
    Positive: 18.83%
    Negative: 48.12%

    Key Topics Identified:
    
Topic 0: bank, im, just, account, love, need, balance, today
Topic 1: holiday, bank, work, weekend, day, good, monday, working
Topic 2: having, account, bank, hi, probs, posting, hes, card

    Risk Insight:
    Elevated negative sentiment around bank may signal
    financial anxiety or trust concerns.

    Confidence Score: 0.66
    


In [21]:
agent.route("Show sentiment summary")


{'positive_ratio': np.float64(0.18833693304535637),
 'negative_ratio': np.float64(0.4812095032397408),
 'neutral_ratio': np.float64(0.3304535637149028),
 'confidence': np.float64(0.6572118323765794)}

In [22]:
agent.route("Explore topics")


[{'topic_id': 0,
  'keywords': ['bank',
   'im',
   'just',
   'account',
   'love',
   'need',
   'balance',
   'today'],
  'examples': ['has to return the shirt she bought from Topshop bc she has $50 in her bank account that has to last her the rest of the month, life sucks ',
   '@NessaBanks Thunder bolts &amp; lightening around here! 5 inches of heavy rain late in the afternoon made it a no go for a road ride.... ']},
 {'topic_id': 1,
  'keywords': ['holiday',
   'bank',
   'work',
   'weekend',
   'day',
   'good',
   'monday',
   'working'],
  'examples': ['spent 1 hour to reach to Axis bank only to find out today is holiday for Mahavir Jayanti  contd..',
   'Working  but looking 4ward 2 thailand and this weeks bank holiday weekend!']},
 {'topic_id': 2,
  'keywords': ['having',
   'account',
   'bank',
   'hi',
   'probs',
   'posting',
   'hes',
   'card'],
  'examples': ['I need to go to the bank tomorrow before I go broke... ',
   "@RoBaBaNks I can't sleep either "]}]

In [23]:
agent.route("Why are people angry at banks?")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,text,distance
596488,why are people that work in banks so silly!!!!,0.542107
901723,@BevClement I don't think we talk much about B...,0.563015
324026,is soooooooooo pissed @ the bank right now,0.623708
312146,"Nationalised banks, worst people to deal with",0.644406
1286362,"does NOT understand banks in the slightest, bu...",0.648075
